# SpectraLite — Colab Bootstrap

Run this **once per new Colab runtime** (A100 / any GPU), then open any phase notebook.

This notebook:
1. Clones or `git pull`s the repo
2. Installs **Colab-safe** deps (`requirements-colab.txt` — does **not** reinstall torch)
3. Verifies CUDA / GPU

After it succeeds, you do **not** need to re-run Phase 0 just to get dependencies.

### 0. Runtime check

Colab menu: **Runtime → Change runtime type → GPU (A100) → Save** before continuing.

In [ ]:
import sys
print("In Colab:", "google.colab" in sys.modules)

try:
    import torch
    print("Torch (before bootstrap):", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: No CUDA yet — select a GPU runtime, then delete runtime if needed.")
except ImportError:
    print("torch not imported yet (ok on a blank machine)")

### 1. Clone or pull repo

Uses the public GitHub repo. Re-run this cell whenever Cursor pushes new phases.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/PrabinDevkota/Execution.git"
CLONE_ROOT = Path("/content/Execution")
PACKAGE_ROOT = CLONE_ROOT / "SpectraLite"

if not PACKAGE_ROOT.is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(CLONE_ROOT)])
else:
    subprocess.check_call(["git", "-C", str(CLONE_ROOT), "pull", "--ff-only"])

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

%cd {PACKAGE_ROOT}
print("Repo ready:", PACKAGE_ROOT)
print("Has requirements-colab.txt:", (PACKAGE_ROOT / "requirements-colab.txt").is_file())
print("Has spectralite/:", (PACKAGE_ROOT / "spectralite").is_dir())

### 2. Install dependencies (Colab-safe)

Installs everything **except** `torch`, so Colab keeps its CUDA build.

In [ ]:
from spectralite.colab_setup import ensure_environment

# pull=False here: we already pulled in the previous cell
repo_root = ensure_environment(pull=False, install=True, require_cuda_on_colab=True)
print("\nBootstrap complete. Open a phase notebook next (Phase0 / Phase1 / ...).")

### 3. Quick import smoke test

Confirms the package imports after install.

In [ ]:
from spectralite import __version__, default_config
from spectralite.system import print_environment_report

print("spectralite", __version__)
print("default model", default_config().model_name)
print_environment_report()
print("\nOK — you can now run phase notebooks in this runtime.")

### Using this from a phase notebook (one cell)

Instead of re-running this whole notebook, any future phase can start with:

```python
import sys
from pathlib import Path
root = Path("/content/Execution/SpectraLite")
if not root.is_dir():
    !git clone https://github.com/PrabinDevkota/Execution.git /content/Execution
sys.path.insert(0, str(root))
from spectralite.colab_setup import ensure_environment
ensure_environment()
```